# My LLM 1: Bigram language model on Shakespeare

**Big picture:** a *character-level* language model. Given one character, predict the next one.
A *bigram* model only looks at the **current** character (no wider context), so it is the simplest
possible baseline before adding self-attention.

Pipeline: text → integers → batches of (input, target) → model → loss → train → sample new text.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F   # F has stateless functions: cross_entropy, softmax, ...

# Fix the random seed so runs are reproducible (same batches, same init, same samples)
torch.manual_seed(1337)

# Load the raw dataset as one big string (~1.1M characters of Shakespeare)
with open('shakespere.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters:", len(text))
print(text[:200])

length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you


## 1. Vocabulary and encode/decode (the tokenizer)

Neural nets work on numbers, not characters. So we build a **tokenizer**:
- `stoi` (string → int) maps each unique character to an id 0..vocab_size-1
- `itos` (int → string) is the reverse map

This is the simplest possible tokenizer (one token = one character). Real LLMs use sub-word
tokenizers (e.g. BPE / tiktoken) with ~50k+ tokens: shorter sequences, but a bigger vocabulary.

In [2]:
# All unique characters in the text, sorted so the mapping is deterministic
chars = sorted(list(set(text)))
vocab_size = len(chars)          # 65 for this dataset
print(''.join(chars))
print("vocab_size:", vocab_size)

# Lookup tables in both directions
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]          # string -> list of ints
decode = lambda l: ''.join([itos[i] for i in l])  # list of ints -> string

# Sanity check: encoding then decoding must give back the original string exactly
s = "hii there"
print(encode(s))
print(decode(encode(s)))
assert decode(encode(s)) == s
assert decode(encode(text)) == text  # round-trip on the whole dataset
print("round-trip OK")


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
vocab_size: 65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there
round-trip OK


## 2. Encode the dataset and do a 90/10 split

- `torch.long` (int64) is required because these ids will be used as **indices** into an embedding table.
- The split is **not shuffled**: the first 90% is train and the last 10% is validation. Val loss then tells
  us how well the model generalises to text it never saw, and is our check for overfitting.

In [3]:
# Whole dataset as one long 1-D tensor of token ids
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

# First 90% -> train, last 10% -> validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print("train:", len(train_data), "val:", len(val_data))

torch.Size([1115394]) torch.int64
train: 1003854 val: 111540


## 3. get_batch: making (x, y) training pairs

We never feed the whole text at once. We sample random **chunks** of length `block_size`.

Key idea: **y is x shifted left by one**. For a chunk `x = data[i : i+T]`, the targets are
`y = data[i+1 : i+T+1]`. So one chunk of length T actually contains **T training examples**:
- context `x[:1]` → target `y[0]`
- context `x[:2]` → target `y[1]`
- ...
- context `x[:T]` → target `y[T-1]`

Shapes: `x` and `y` are both `(B, T)`, where B = batch size (parallel sequences) and T = time/context length.

In [4]:
batch_size = 32  # B: how many independent sequences we process in parallel
block_size = 8   # T: maximum context length for predictions

def get_batch(split):
    d = train_data if split == 'train' else val_data
    # B random starting positions; "- block_size" so that i+block_size+1 stays in range
    ix = torch.randint(len(d) - block_size, (batch_size,))
    x = torch.stack([d[i:i+block_size] for i in ix])        # (B, T) inputs
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])    # (B, T) targets = inputs shifted by 1
    return x, y

xb, yb = get_batch('train')
print("x shape:", xb.shape, "y shape:", yb.shape)
print("x[0]:", xb[0].tolist())
print("y[0]:", yb[0].tolist())
assert torch.equal(xb[0, 1:], yb[0, :-1])  # proves the shift-by-one

# Unroll one row to see the T examples hidden inside it
for t in range(block_size):
    context = xb[0, :t+1]
    target = yb[0, t]
    print(f"when input is {context.tolist()} the target is {target.item()}")

x shape: torch.Size([32, 8]) y shape: torch.Size([32, 8])
x[0]: [24, 43, 58, 5, 57, 1, 46, 43]
y[0]: [43, 58, 5, 57, 1, 46, 43, 39]
when input is [24] the target is 43
when input is [24, 43] the target is 58
when input is [24, 43, 58] the target is 5
when input is [24, 43, 58, 5] the target is 57
when input is [24, 43, 58, 5, 57] the target is 1
when input is [24, 43, 58, 5, 57, 1] the target is 46
when input is [24, 43, 58, 5, 57, 1, 46] the target is 43
when input is [24, 43, 58, 5, 57, 1, 46, 43] the target is 39


## 4. BigramLanguageModel

The whole model is **one lookup table** of shape `(vocab_size, vocab_size)`.
Row `i` holds the **logits** (unnormalised scores) for "which character comes after character i".

- `nn.Embedding(vocab_size, vocab_size)` is a learnable table. Indexing it with token ids picks out rows.
- Input `idx (B, T)` gives `logits (B, T, C)` where C = vocab_size.
- **Loss** = cross-entropy between the predicted distribution and the true next character.
  `F.cross_entropy` expects `(N, C)` logits and `(N,)` targets, so we flatten B and T together into N = B*T.
- `targets=None` (used during generation) means there is nothing to compare against, so `loss = None`.

**Generation** (autoregressive sampling): take the logits for the *last* position, softmax them into
probabilities, sample one token, append it to the sequence, and repeat.

In [5]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table.
        # Row i = scores for "what follows token i". This is the ONLY parameter: 65*65 = 4225 numbers.
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: (B, T) integer token ids
        logits = self.token_embedding_table(idx)  # (B, T, C): one row of scores per position

        if targets is None:
            loss = None                           # inference mode: nothing to score against
        else:
            B, T, C = logits.shape
            # cross_entropy wants (N, C) and (N,), so merge batch and time dimensions
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)  # = mean of -log(prob of correct next char)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, _ = self(idx)                         # (B, T, C)
            logits = logits[:, -1, :]                     # (B, C): only the last time step matters for a bigram
            probs = F.softmax(logits, dim=-1)             # (B, C): scores -> probabilities summing to 1
            idx_next = torch.multinomial(probs, num_samples=1)  # (B, 1): SAMPLE (not argmax) so output varies
            idx = torch.cat((idx, idx_next), dim=1)       # (B, T+1): append and feed back in
        return idx

model = BigramLanguageModel(vocab_size)
logits, loss = model(xb, yb)
print(logits.shape)   # (B*T, C) = (256, 65) because forward reshaped it for the loss
print("initial loss:", loss.item())
# If every char were equally likely, p = 1/65, so loss = -ln(1/65) = ln(65)
print("expected  -ln(1/vocab_size):", torch.log(torch.tensor(float(vocab_size))).item())

# Calling without targets must return loss = None
_, no_loss = model(xb)
print("loss without targets:", no_loss)

torch.Size([256, 65])
initial loss: 4.692231178283691
expected  -ln(1/vocab_size): 4.174387454986572
loss without targets: None


**Why isn't the initial loss exactly 4.17?** The expected loss with uniform predictions is -ln(1/65) ≈ 4.17.
The measured initial loss is higher (~4.7) because `nn.Embedding` initialises its weights from N(0, 1), so the
starting logits aren't perfectly uniform. The model starts out slightly *confidently wrong*.
Zero-initialising the table (`self.token_embedding_table.weight.data.zero_()`) would give exactly 4.17.

**Revision tip:** always check the initial loss against `ln(vocab_size)`. A much larger value means the
initialisation is bad. A value far below it means something is leaking (e.g. targets in the input).

## 5. Generate before training (should be pure noise)

In [6]:
# Start from token 0 (which is '\n') as the initial context, shape (1, 1) = batch of 1, length 1
context = torch.zeros((1, 1), dtype=torch.long)
# generate returns (1, 301); take row 0, convert to a Python list, decode to a string
print(decode(model.generate(context, max_new_tokens=300)[0].tolist()))


sV
vL
ja,FsLY,wxEuS'pao3jOssyBA$zFqYTkeMk x-gQ.FzLg!iKI.egzDnyA TsTbvdgX!KpGIeJyjv,SrFF&SDt!:hwWSl.WxbjOQDokT.ry,;vWr$Z-JX$WSeJDKbZ-g,xOSKYw?Zfyf&w WlAlABvtX$LePM!RFUcaMWlAv,p:Zfsy-QlkNjxfjqgoqX-AG'L.z$$lNRYV
QsLVLempRQ!OSZ'kNGbsFMWy'hjCchM.W.j&PlA
Q.x?Dysdx
?'OC?KjOHHYxG&DotWusyREMJFzqX&FgUNw$iEQzY


## 6. Train

The standard PyTorch training loop:
1. sample a batch
2. forward pass → loss
3. `zero_grad` (clear old gradients, because PyTorch *accumulates* them otherwise)
4. `loss.backward()` (compute gradients)
5. `optimizer.step()` (update the weights)

`estimate_loss` averages the loss over many batches, so the printed numbers are smooth instead of
noisy single-batch values. If train ≪ val, the model is overfitting.

In [7]:
learning_rate = 1e-3
max_iters = 5000
eval_interval = 500   # how often to print train/val loss
eval_iters = 200      # how many batches to average over for each estimate

@torch.no_grad()      # no gradients needed for evaluation: faster and uses less memory
def estimate_loss():
    out = {}
    model.eval()      # switch to eval mode (matters later for dropout/batchnorm; a no-op here)
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean().item()
    model.train()     # back to training mode
    return out

# AdamW = Adam + decoupled weight decay; the usual default for transformers
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for step in range(max_iters):
    # Periodically report averaged train/val loss
    if step % eval_interval == 0 or step == max_iters - 1:
        losses = estimate_loss()
        print(f"step {step}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')               # 1. sample a batch
    logits, loss = model(xb, yb)              # 2. forward pass
    optimizer.zero_grad(set_to_none=True)     # 3. clear old gradients
    loss.backward()                           # 4. backprop
    optimizer.step()                          # 5. update parameters

step 0: train loss 4.7669, val loss 4.7682


step 500: train loss 4.2249, val loss 4.2328


step 1000: train loss 3.7742, val loss 3.7823


step 1500: train loss 3.4226, val loss 3.4388


step 2000: train loss 3.1600, val loss 3.1717


...

step 3000: train loss 2.8216, val loss 2.8342


step 3500: train loss 2.7288, val loss 2.7346


step 4000: train loss 2.6588, val loss 2.6567


step 4500: train loss 2.6105, val loss 2.6207


step 4999: train loss 2.5711, val loss 2.5911


## 7. Generate after training

Expect text that *looks* like Shakespeare (line breaks, capitalised speaker names, short word-like
chunks) but makes no sense. The model only ever sees **one** previous character.
Loss plateaus around ~2.5 for a bigram model. To do better, tokens need to communicate with earlier
tokens, which is exactly what **self-attention** adds next.

In [8]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


TiCHishadjeyQYO al thangjENLINZWogqus
Inengheeve YDx'-Angereco, KNGorEO, thent je retahe s d baMoNst fine smir hew.
Gove!
ING hiKIFFzexEEno s thMEGoveis menorthardellSe, fakngamy ortsouBERPr, le, d asote meisy haystan; wnoavethJf hanste, ut
I he.
Whiutt t
vveen. mh tsLhee datheJzL:
AMAMinou'GLepr-leseres dend w hinn n merdu:
Anm r, noromaprisear n d mand n we deeP o bpith'deymeld a. blimst.x?O:
I celd
IFrie-ve h: re, s, prceresoor the eZfourer kJJo3? vay, t mes hiskl, tinanrbol cthe hesighated;p
